# Validation

In [1]:
import pandas as pd
import numpy as np

In [2]:
pd.set_option('display.max_colwidth', None)

In [3]:
# Paths
PATH = '/home/tatiane/repositorios/brazil-health-data-project/data_pipeline'
raw_path = f"{PATH}/data/raw/ibge_populacao/tabela6579.csv"
standardization_path = f"{PATH}/data/processed/ibge_populacao/ibge_populacao2.parquet"

In [4]:
# Reading
raw = pd.read_csv(raw_path, sep=";", skiprows=3)
standardization = pd.read_parquet(standardization_path)

- Validation

In [5]:
resumo_raw_wide = pd.DataFrame({
    'tipo': raw.dtypes,
    'nulos': raw.isnull().sum(),
    'perc_nulos': (raw.isnull().sum() / len(raw)) * 100,
    'unicos': raw.nunique()
})

resumo_raw_wide

,tipo,nulos,perc_nulos,unicos
Cód.,object,1,0.017319,5761
Unidade da Federação e Município,object,169,2.926914,5605
2018,object,176,3.048147,5122
2019,object,176,3.048147,5104
2020,object,176,3.048147,5150
2021,object,176,3.048147,5137


In [6]:
resumo_standardization_long = pd.DataFrame({
    'tipo': standardization.dtypes,
    'nulos': standardization.isnull().sum(),
    'perc_nulos': (standardization.isnull().sum() / len(standardization)) * 100,
    'unicos': standardization.nunique()
})

resumo_standardization_long

,tipo,nulos,perc_nulos,unicos
municipio,object,0,0.000000,5318
uf,object,112,0.500089,27
cod_uf,Int64,4,0.017860,27
ano,Int64,0,0.000000,4
populacao,Int64,8,0.035721,16633
cod_municipio,Int64,112,0.500089,5571


### Destrinchando

1. Quantidade de linhas esperadas

In [9]:
num_municipios = raw.shape[0]
num_anos = 4

expected_rows = num_municipios * num_anos
actual_rows = standardization.shape[0]

print(expected_rows, actual_rows)

23096 22396


2. Verificando se todos os anos existem

In [11]:
anos_esperados = {2018, 2019, 2020, 2021}
anos_long = set(standardization["ano"].unique())

assert anos_esperados == anos_long

3. Validando duplicidade

In [12]:
standardization["nivel_geografico"] = np.select(
    [
        standardization["cod_municipio"].notnull(),
        standardization["cod_uf"].notnull() & standardization["uf"].isnull()
    ],
    [
        "municipio",
        "estado"
    ],
    default="outros"
)

In [13]:
duplicados_estado = (
    standardization
    .query("nivel_geografico == 'estado'")
    .duplicated(
        subset=["cod_uf", "ano"]
    )
    .sum()
)

print(duplicados_estado)

0


4. Validação  da preservação dos valores

In [20]:
raw[raw.eq("...").any(axis=1)]

,Cód.,Unidade da Federação e Município,2018,2019,2020,2021
5226,5101837,Boa Esperança do Norte (MT),...,...,...,...
5772,...,Valor não disponível.\nEx: A produção de feijão em determinado município não foi pesquisada ou determinado município não existia no ano da pesquisa.,NaN,NaN,NaN,NaN


In [21]:
standardization[standardization.eq("...").any(axis=1)]

,municipio,uf,cod_uf,ano,populacao,cod_municipio,nivel_geografico


In [22]:
raw = raw.replace("...", pd.NA)
for ano in anos:
    raw[ano] = pd.to_numeric(raw[ano], errors="coerce")

In [25]:
# anos da base
anos = ["2018", "2019", "2020", "2021"]

# substitui "..." por nulo
raw = raw.replace("...", pd.NA)

# converte colunas dos anos para numérico
for ano in anos:
    raw[ano] = pd.to_numeric(raw[ano], errors="coerce")

# lista para armazenar erros
erros = []

# loop de validação
for _, linha in raw.iterrows():

    codigo = str(linha["Cód."])
    nome = linha["Unidade da Federação e Município"]

    # identifica estado vs município
    is_estado = len(codigo) == 2

    for ano in anos:

        valor_raw = linha[ano]

        # ignora valores nulos
        if pd.isna(valor_raw):
            continue

        # busca no dataframe padronizado
        if is_estado:

            resultado = standardization[
                (standardization["cod_uf"].astype(str) == codigo) &
                (standardization["ano"].astype(str) == ano) &
                (standardization["nivel_geografico"] == "estado")
            ]

        else:

            resultado = standardization[
                (standardization["cod_municipio"].astype(str) == codigo) &
                (standardization["ano"].astype(str) == ano) &
                (standardization["nivel_geografico"] == "municipio")
            ]

        # valida existência
        if resultado.empty:

            erros.append({
                "codigo": codigo,
                "nome": nome,
                "ano": ano,
                "erro": "registro não encontrado"
            })

            continue

        valor_long = resultado["populacao"].iloc[0]

        # valida valor
        if valor_raw != valor_long:

            erros.append({
                "codigo": codigo,
                "nome": nome,
                "ano": ano,
                "valor_raw": valor_raw,
                "valor_long": valor_long,
                "erro": "valor divergente"
            })

In [26]:
erros_df = pd.DataFrame(erros)

print(erros_df.head())
print(f"Total de erros: {len(erros_df)}")

Empty DataFrame
Columns: []
Index: []
Total de erros: 0


In [27]:
raw.head(500)

,Cód.,Unidade da Federação e Município,2018,2019,2020,2021
0,11,Rondônia,1757589.0,1777225.0,1796460.0,1815278.0
1,12,Acre,869265.0,881935.0,894470.0,906876.0
2,13,Amazonas,4080611.0,4144597.0,4207714.0,4269995.0
3,14,Roraima,576568.0,605761.0,631181.0,652713.0
4,15,Pará,8513497.0,8602865.0,8690745.0,8777124.0
...,...,...,...,...,...,...
495,2101103,Axixá (MA),12076.0,12130.0,12183.0,12234.0
496,2101202,Bacabal (MA),104633.0,104949.0,104790.0,105094.0
497,2101251,Bacabeira (MA),16855.0,17055.0,17252.0,17446.0
498,2101301,Bacuri (MA),18508.0,18582.0,18654.0,18726.0


In [30]:
standardization[
    (standardization["cod_municipio"] == '2101251')
].head()

,municipio,uf,cod_uf,ano,populacao,cod_municipio,nivel_geografico
497,Bacabeira,MA,21,2018,16855,2101251,municipio
6271,Bacabeira,MA,21,2019,17055,2101251,municipio
12045,Bacabeira,MA,21,2020,17252,2101251,municipio
17819,Bacabeira,MA,21,2021,17446,2101251,municipio


5. Validando a soma total

In [31]:
# soma enquanto wide
soma_wide = (
    raw[["2018", "2019", "2020", "2021"]]
    .sum()
    .sum()
)

In [32]:
soma_long = standardization["populacao"].sum()

In [33]:
assert soma_wide == soma_long

6. Validar cardinalidade por município

In [34]:
validacao_cardinalidade = (
    standardization.groupby("cod_municipio")
    .size()
    .value_counts()
)

print(validacao_cardinalidade)

4    5571
Name: count, dtype: int64
